In [1]:
# Databricks notebook source
# MAGIC %md This notebook is slightly modified version of `MLflow Training Tutorial` from [MLflow examples](https://github.com/mlflow/mlflow/tree/master/examples/sklearn_elasticnet_wine).
# MAGIC
# MAGIC It predicts the quality of wine using [sklearn.linear_model.ElasticNet](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html).
# MAGIC This is a base code and will be modified further during the `Databricks: Reproducible experiments with MLflow and Delta Lake` tutorial.
# MAGIC
# MAGIC Attribution
# MAGIC * The data set used in this example is from http://archive.ics.uci.edu/ml/datasets/Wine+Quality
# MAGIC * P. Cortez, A. Cerdeira, F. Almeida, T. Matos and J. Reis.
# MAGIC * Modeling wine preferences by data mining from physicochemical properties. In Decision Support Systems, Elsevier, 47(4):547-553, 2009.

In [15]:
import os
import warnings
import sys

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet

import logging

import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestRegressor

from mlflow.models.signature import infer_signature


In [19]:

def train():
    logging.basicConfig(level=logging.WARN)
    logger = logging.getLogger(__name__)

    def eval_metrics(actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2

    warnings.filterwarnings("ignore")
    np.random.seed(40)

    # Charger les données
    csv_url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
    try:
        data = pd.read_csv(csv_url, sep=';')
    except Exception as e:
        logger.exception(
            "Unable to download training & test CSV, check your internet connection. Error: %s", e)

    # Split en train/test
    train, test = train_test_split(data)
    train_x = train.drop(["quality"], axis=1)
    test_x = test.drop(["quality"], axis=1)
    train_y = train[["quality"]]
    test_y = test[["quality"]]

    # Définir le nom d'expérience
    mlflow.set_experiment("ElasticNet-Wine-Tracking")

    # Boucle sur les hyperparamètres
    for alpha in [0.1, 0.5, 1.0]:
        for l1_ratio in [0.1, 0.5, 0.9]:
            run_name = f"ENet (α={alpha}, l1={l1_ratio})"
            with mlflow.start_run(run_name=run_name):
                model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42)
                model.fit(train_x, train_y)

                predicted_qualities = model.predict(test_x)
                (rmse, mae, r2) = eval_metrics(test_y, predicted_qualities)

                # Log params, metrics & model
                mlflow.log_param("alpha", alpha)
                mlflow.log_param("l1_ratio", l1_ratio)
                mlflow.log_param("model_type", "ElasticNet")
                mlflow.log_metric("rmse", rmse)
                mlflow.log_metric("mae", mae)
                mlflow.log_metric("r2", r2)
                mlflow.sklearn.log_model(model, "model", input_example=train_x.head(1))

    # Affichage des dernières métriques
    predicted_qualities = model.predict(test_x)
    (rmse, mae, r2) = eval_metrics(test_y, predicted_qualities)

    print(f"ElasticNet model (alpha={alpha}, l1_ratio={l1_ratio}):")
    print("  RMSE:", rmse)
    print("  MAE:", mae)
    print("  R2:", r2)

    with open("metrics.txt", 'w') as f:
        f.write(f"  RMSE: {rmse}\n")
        f.write(f"  MAE: {mae}\n")
        f.write(f"  R2: {r2}\n")

In [21]:
train()

2025/03/24 11:52:59 INFO mlflow.tracking.fluent: Experiment with name 'ElasticNet-Wine-Tracking' does not exist. Creating a new experiment.


ElasticNet model (alpha=1.0, l1_ratio=0.9):
  RMSE: 0.8328836770764705
  MAE: 0.6717584996142674
  R2: 0.017115625319085725


In [17]:
# Wine Quality Sample
def train():
    logging.basicConfig(level=logging.WARN)
    logger = logging.getLogger(__name__)

    def eval_metrics(actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2

    warnings.filterwarnings("ignore")
    np.random.seed(40)

    # Read the wine-quality csv file from the URL
    csv_url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
    try:
        data = pd.read_csv(csv_url, sep=';')
    except Exception as e:
        logger.exception(
            "Unable to download training & test CSV, check your internet connection. Error: %s", e)

    # Split the data into training and test sets
    train, test = train_test_split(data)
    train_x = train.drop(["quality"], axis=1)
    test_x = test.drop(["quality"], axis=1)
    train_y = train[["quality"]]
    test_y = test[["quality"]]

    mlflow.set_experiment("RandomForest-Wine-Tracking")

    # ------------------- Model tracking -------------------
    for n_estimators in [50, 100, 200]:
        for max_depth in [5, 10, None]:
            run_name = f"RF (n={n_estimators}, depth={max_depth})"
            with mlflow.start_run(run_name=run_name):
                rf = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
                rf.fit(train_x, train_y.values.ravel())
                predicted_qualities = rf.predict(test_x)
                (rmse, mae, r2) = eval_metrics(test_y, predicted_qualities)

                # Signature + input sample
                signature = infer_signature(train_x, rf.predict(train_x))
                input_example = train_x.iloc[:5]

                # Log params, metrics, and model
                mlflow.log_param("n_estimators", n_estimators)
                mlflow.log_param("max_depth", max_depth)
                mlflow.log_param("model_type", "RandomForest")
                mlflow.log_metric("rmse", rmse)
                mlflow.log_metric("mae", mae)
                mlflow.log_metric("r2", r2)
                mlflow.sklearn.log_model(
                    rf,
                    "model",
                    signature=signature,
                    input_example=input_example
                )

    # ------------------ Final report (last model) ------------------
    predicted_qualities = rf.predict(test_x)
    (rmse, mae, r2) = eval_metrics(test_y, predicted_qualities)

    print("Random Forest model (n_estimators=%d, max_depth=%s):" % (n_estimators, str(max_depth)))
    print("  RMSE: %s" % rmse)
    print("  MAE: %s" % mae)
    print("  R2: %s" % r2)

    with open("metrics.txt", 'w') as outfile:
        outfile.write("  RMSE: %s\n" % rmse)
        outfile.write("  MAE: %s\n" % mae)
        outfile.write("  R2: %s\n" % r2)


In [18]:
# Start the training
#train(0.3, 0.8)
train()

2025/03/24 11:48:23 INFO mlflow.tracking.fluent: Experiment with name 'RandomForest-Wine-Tracking' does not exist. Creating a new experiment.


Random Forest model (n_estimators=200, max_depth=None):
  RMSE: 0.5783625052681061
  MAE: 0.41566250000000005
  R2: 0.5260484042364777
